# Model Inference Pipeline
Objective: Load the serialized `.joblib` models and execute predictions on new, unlabelled data efficiently.

### 1. Kesiapan Inferensi & Manajemen Dependensi
Ensure the environment matches `requirements.txt`. This script is designed to run locally or in a containerized environment with minimal RAM usage.

In [ ]:
import pandas as pd
import joblib
import os

print("1. Loading Serialized Models...")
try:
    prophet_model = joblib.load('Models/prophet_model.joblib')
    lgbm_model = joblib.load('Models/lgbm_model.joblib')
    iso_forest = joblib.load('Models/iso_forest.joblib')
    print("Models loaded successfully.")
except FileNotFoundError:
    print("Error: Models not found. Please run training.ipynb first.")

### 2. Eksekusi Prediksi
Processing the incoming data through the Prophet baseline and LightGBM residual corrector, while checking for structural grid anomalies.

In [ ]:
print("2. Loading New Data for Inference...")
# In a real scenario, this would be new, unseen data. 
# We are using the processed dataset here to demonstrate the pipeline.
new_data_path = 'Outputs/dataset_daily_processed.csv' 
df_new = pd.read_csv(new_data_path)
df_new['Date'] = pd.to_datetime(df_new['Date'])

# Define required schema
required_features = [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday', 'Avg_Temp', 'Rainfall', 
    'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_7', 'GDP', 'Population', 'Industrial_Index'
]

# Ensure required columns exist (mocking the macro merge if missing in this CSV)
for col in required_features:
    if col not in df_new.columns:
        df_new[col] = 0 # Fallback for inference safety

# Drop NaNs to prevent inference crashes
df_clean = df_new.dropna(subset=required_features).copy()

print("3. Generating Hybrid Predictions...")
# Step A: Prophet Baseline
df_prophet = df_clean[['Date']].rename(columns={'Date': 'ds'})
prophet_preds = prophet_model.predict(df_prophet)['yhat'].values

# Step B: LightGBM Residuals
lgbm_preds = lgbm_model.predict(df_clean[required_features])

# Step C: Anomaly Flagging
df_clean['Anomaly_Flag'] = iso_forest.predict(df_clean[required_features])
df_clean['Anomaly_Status'] = df_clean['Anomaly_Flag'].apply(lambda x: 'ANOMALY' if x == -1 else 'NORMAL')

# Final Output
df_clean['Final_Predicted_Demand'] = prophet_preds + lgbm_preds

# Display output ready for API or Dashboard
print("\n--- INFERENCE RESULTS (Sample) ---")
display(df_clean[['Date', 'Final_Predicted_Demand', 'Anomaly_Status']].tail(5))

# Save inference results
df_clean.to_csv('Outputs/inference_results.csv', index=False)
print("\n Inference complete. Results saved to Outputs/inference_results.csv")